# 01 – Kom i gang med Slettix Analytics

Denne notebooken demonstrerer:
- Opprette en SparkSession koblet til den lokale Spark-clusteren
- Grunnleggende DataFrame-operasjoner
- Skrive og lese en Delta-tabell på MinIO


## 1. Opprett SparkSession

`spark-defaults.conf` er montert inn og konfigurerer Delta Lake og S3A/MinIO automatisk.  
Vi trenger kun å spesifisere master-URL-en.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("01-getting-started")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark versjon : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 20:12:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark versjon : 3.5.8
Master        : spark://spark-master:7077


## 2. Grunnleggende DataFrame-operasjoner

In [2]:
from datetime import date

data = [
    (1, "Alice",   "engineering", 95000, date(2022, 3, 1)),
    (2, "Bob",     "marketing",   72000, date(2021, 7, 15)),
    (3, "Charlie", "engineering", 105000, date(2020, 1, 10)),
    (4, "Diana",   "marketing",   68000, date(2023, 5, 20)),
    (5, "Eve",     "engineering", 98000, date(2019, 11, 3)),
]

df = spark.createDataFrame(data, ["id", "name", "department", "salary", "hire_date"])
df.printSchema()
df.show()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- hire_date: date (nullable = true)



+---+-------+-----------+------+----------+
| id|   name| department|salary| hire_date|
+---+-------+-----------+------+----------+
|  1|  Alice|engineering| 95000|2022-03-01|
|  2|    Bob|  marketing| 72000|2021-07-15|
|  3|Charlie|engineering|105000|2020-01-10|
|  4|  Diana|  marketing| 68000|2023-05-20|
|  5|    Eve|engineering| 98000|2019-11-03|
+---+-------+-----------+------+----------+



In [3]:
# Gjennomsnittlig lønn per avdeling
df.groupBy("department") \
  .agg(
      F.count("*").alias("antall"),
      F.avg("salary").alias("gj_snitt_lønn"),
      F.max("salary").alias("maks_lønn")
  ) \
  .orderBy("department") \
  .show()

+-----------+------+-----------------+---------+
| department|antall|    gj_snitt_lønn|maks_lønn|
+-----------+------+-----------------+---------+
|engineering|     3|99333.33333333333|   105000|
|  marketing|     2|          70000.0|    72000|
+-----------+------+-----------------+---------+



## 3. Skriv til Delta-tabell på MinIO (Bronze-lag)

In [4]:
TARGET_PATH = "s3a://bronze/employees"

df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(TARGET_PATH)

print(f"Skrev {df.count()} rader til {TARGET_PATH}")

26/03/24 20:13:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Skrev 5 rader til s3a://bronze/employees


## 4. Les tilbake fra Delta-tabell

In [5]:
df_delta = spark.read.format("delta").load(TARGET_PATH)

print(f"Rader lest: {df_delta.count()}")
df_delta.show()

26/03/24 20:13:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Rader lest: 5
+---+-------+-----------+------+----------+
| id|   name| department|salary| hire_date|
+---+-------+-----------+------+----------+
|  3|Charlie|engineering|105000|2020-01-10|
|  4|  Diana|  marketing| 68000|2023-05-20|
|  5|    Eve|engineering| 98000|2019-11-03|
|  1|  Alice|engineering| 95000|2022-03-01|
|  2|    Bob|  marketing| 72000|2021-07-15|
+---+-------+-----------+------+----------+



## 5. Inspiser Delta-tabellens historikk

Delta Lake loggfører alle operasjoner. Her ser vi versjonshistorikken.

In [6]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, TARGET_PATH)
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-------------------+---------+--------------------------------------+
|version|timestamp          |operation|operationParameters                   |
+-------+-------------------+---------+--------------------------------------+
|0      |2026-03-24 20:13:31|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-------------------+---------+--------------------------------------+



In [ ]:
spark.stop()